In [2]:
import os

In [5]:
%pwd

'/Users/artemvardanan/CV_end2end/CNN-end2end'

In [4]:
os.chdir("../")

In [11]:
!pip install dagshub

In [12]:
!pip install ipywidgets

In [6]:
import dagshub
dagshub.init(repo_owner='Artom121', repo_name='CNN-end2end', mlflow=True)

# import mlflow
# with mlflow.start_run():
#   mlflow.log_param('parameter name', 'value')
#   mlflow.log_metric('metric name', 1)

Accessing as Artom121

Initialized MLflow to track repo "Artom121/CNN-end2end"

Repository Artom121/CNN-end2end initialized!

In [15]:
import tensorflow as tf

In [16]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [18]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [19]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model = 'artifacts/training/model.h5',
            training_data = 'artifacts/data_ingestion/kidney-ct-scan-image',
            mlflow_uri = 'https://dagshub.com/Artom121/CNN-end2end.mlflow',
            all_params = self.params,
            params_image_size = self.params.IMAGE_SIZE,
            params_batch_size = self.params.BATCH_SIZE,
        )
        return eval_config

In [20]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse
import keras

In [21]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation='bilinear'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset='validation',
            shuffle=False,
            **dataflow_kwargs
        )
    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        # self.score = model.evaluate(self.valid_generator)
        self.score = self.model.evaluate(self.valid_generator, return_dict=True)
        self.save_score()
    
    def save_score(self):
        # scores = {'loss': self.score[0], 'accuracy': self.score[1]}
        save_json(path=Path("scores.json"), data=self.score)
    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(self.score)

            if tracking_url_type_store != 'file':
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")

In [22]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
    raise e

[2026-09-02 05:47:31,353: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-09-02 05:47:31,355: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-02 05:47:31,355: INFO: common: created directory at: artifacts]
Found 139 images belonging to 2 classes.
9/9 [==============================] - 8s 925ms/step - loss: 14.6125 - accuracy: 0.7770
[2026-09-02 05:47:40,005: INFO: common: json file saved at: scores.json]


2026/09/02 05:47:41 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /var/folders/z6/14mr22v51v5871tz8z3f2wr40000gn/T/tmpx9q_6den/model/data/model/assets
[2026-09-02 05:47:42,093: INFO: builder_impl: Assets written to: /var/folders/z6/14mr22v51v5871tz8z3f2wr40000gn/T/tmpx9q_6den/model/data/model/assets]


Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/09/02 05:48:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 4
Created version '4' of model 'VGG16Model'.
